In [3]:
!pip install --upgrade pip
!pip install torch-summary
!pip install arabert peft transformers pytorch_lightning
!pip install git+https://github.com/ARBML/rouge_score_ar
!pip install accelerate -U
!pip install sentencepiece 
!pip install lxml 
!pip install keras-nlp
!pip install gdown
!pip install bitsandbytes peft
!pip install datasets

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 25.8 MB/s eta 0:00:0000:0100:01
  Attempting uninstall: pip
    Found existing installation: pip 23.0.1
    Uninstalling pip-23.0.1:
      Successfully uninstalled pip-23.0.1
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 4.6 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.8/56.8 kB 5.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 58.1 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 721.2/721.2 kB 35.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.4/126.4 kB 11.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.0/185.0 kB 17.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 227.6/227.6 kB 19.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.8/268.8 kB 22.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 772.3/772.3 kB 31.2

In [8]:
def map_to(batch):

    model_inputs = tokenizer(batch['article'],
                             padding="max_length",
                             truncation=True,
                              max_length=900, return_tensors='np')

    labels = tokenizer(batch['summary'], text_target=batch['summary'],
                       padding="max_length",
                       truncation=True,
                       max_length=300, return_tensors='np')
    
    model_inputs['decoder_input_ids'] = labels['input_ids']
    model_inputs['decoder_attention_mask'] = labels['attention_mask']
    model_inputs["labels"] = labels["labels"]
    
    return model_inputs


In [5]:
import tensorflow as tf

from transformers import AutoTokenizer, TFAutoModelForSeq2SeqLM
from datasets import load_dataset, DatasetDict
from transformers import DataCollatorForSeq2Seq
from keras_nlp.metrics import Perplexity

D0707 21:10:19.842992537      15 config.cc:119]                        gRPC EXPERIMENT tcp_frame_size_tuning               OFF (default:OFF)
D0707 21:10:19.843020436      15 config.cc:119]                        gRPC EXPERIMENT tcp_rcv_lowat                       OFF (default:OFF)
D0707 21:10:19.843023722      15 config.cc:119]                        gRPC EXPERIMENT peer_state_based_framing            OFF (default:OFF)
D0707 21:10:19.843026259      15 config.cc:119]                        gRPC EXPERIMENT flow_control_fixes                  ON  (default:ON)
D0707 21:10:19.843028596      15 config.cc:119]                        gRPC EXPERIMENT memory_pressure_controller          OFF (default:OFF)
D0707 21:10:19.843031152      15 config.cc:119]                        gRPC EXPERIMENT unconstrained_max_quota_buffer_size OFF (default:OFF)
D0707 21:10:19.843033502      15 config.cc:119]                        gRPC EXPERIMENT new_hpack_huffman_decoder           ON  (default:ON)
D0707 21:10:19.

In [6]:

tpu = tf.distribute.cluster_resolver.TPUClusterResolver.connect(tpu="local") # "local" for 1VM TPU

strategy = tf.distribute.TPUStrategy(tpu)
print("RUNNING TPU")


print("REPLICAS: ", strategy.num_replicas_in_sync)

INFO:tensorflow:Deallocate tpu buffers before initializing tpu system.
INFO:tensorflow:Initializing the TPU system: local
INFO:tensorflow:Finished initializing TPU system.
INFO:tensorflow:Found TPU system:
INFO:tensorflow:*** Num TPU Cores: 8
INFO:tensorflow:*** Num TPU Workers: 1
INFO:tensorflow:*** Num TPU Cores Per Worker: 8
INFO:tensorflow:*** Available Device: _DeviceAttributes(/job:localhost/replica:0/task:0/device:CPU:0, CPU, 0, 0)
INFO:tensorflow:*** Available Device: _DeviceAttributes(/job:localhost/replica:0/task:0/device:TPU:0, TPU, 0, 0)
INFO:tensorflow:*** Available Device: _DeviceAttributes(/job:localhost/replica:0/task:0/device:TPU:1, TPU, 0, 0)
INFO:tensorflow:*** Available Device: _DeviceAttributes(/job:localhost/replica:0/task:0/device:TPU:2, TPU, 0, 0)
INFO:tensorflow:*** Available Device: _DeviceAttributes(/job:localhost/replica:0/task:0/device:TPU:3, TPU, 0, 0)
INFO:tensorflow:*** Available Device: _DeviceAttributes(/job:localhost/replica:0/task:0/device:TPU:4, TPU

In [9]:
model_id = "abdalrahmanshahrour/arabartsummarization"

with strategy.scope():
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = TFAutoModelForSeq2SeqLM.from_pretrained(model_id, from_pt=True)
    opt = tf.keras.optimizers.AdamW(learning_rate=1e-5, weight_decay=0.4, amsgrad=True)
    model.compile(optimizer=opt, metrics=[Perplexity(from_logits=True, mask_token_id=tokenizer.pad_token_id, name="perplexity")])


Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFMBartForConditionalGeneration: ['lm_head.weight']
- This IS expected if you are initializing TFMBartForConditionalGeneration from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFMBartForConditionalGeneration from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model from a BertForSequenceClassification model).
All the weights of TFMBartForConditionalGeneration were initialized from the PyTorch model.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFMBartForConditionalGeneration for predictions without further training.


In [10]:
import os
path = '/kaggle/input/arabictextsummarization'
train_files = {'train': [
                         '/kaggle/input/arabictextsummarization/EGY_alyoomSabi_news_1.csv',
                         '/kaggle/input/arabictextsummarization/PAL_alquds_uk_news.csv',
                         '/kaggle/input/arabictextsummarization/نسخة من SAU_ryiadh_news_2.csv'
                        ],
               
               'validation': ['/kaggle/input/arabictextsummarization/MUR_sahra_news.csv',
                             ]
              }

In [11]:
data = load_dataset('csv', data_files=train_files)



Extracting data files: 100%|██████████| 2/2 [00:00<00:00, 18.76it/s]

Generating train split: 0 examples [00:00, ? examples/s]
Generating train split: 10000 examples [00:00, 20236.70 examples/s]
Generating train split: 20000 examples [00:00, 20485.87 examples/s]
Generating train split: 30000 examples [00:01, 21242.70 examples/s]
Generating train split: 40000 examples [00:01, 21461.96 examples/s]
Generating train split: 50000 examples [00:02, 21138.12 examples/s]
Generating train split: 60000 examples [00:02, 21176.93 examples/s]
Generating train split: 70000 examples [00:03, 21006.41 examples/s]
Generating train split: 80000 examples [00:03, 20637.50 examples/s]
Generating train split: 90000 examples [00:04, 20061.50 examples/s]
Generating train split: 100000 examples [00:04, 19653.44 examples/s]
Generating train split: 110000 examples [00:05, 19261.11 examples/s]
Generating train split: 120000 examples [00:05, 19182.71 examples/s]
Generating train split: 130000 examples [00:06, 1883

Dataset csv downloaded and prepared to /root/.cache/huggingface/datasets/csv/default-5de9a8e0a291a618/0.0.0/eea64c71ca8b46dd3f537ed218fc9bf495d5707789152eb2764f5c78fa66d59d. Subsequent calls will reuse this data.



100%|██████████| 2/2 [00:00<00:00, 48.63it/s]


In [12]:
data

DatasetDict({
    train: Dataset({
        features: ['item', 'address', 'title', 'article', 'summary'],
        num_rows: 1491651
    })
    validation: Dataset({
        features: ['item', 'address', 'title', 'article', 'summary'],
        num_rows: 11983
    })
})

In [13]:
train_dataset = data['train'].shuffle(seed=36).map(map_to, batched=True, batch_size=1000, drop_last_batch=True, cache_file_name='train.pkl', load_from_cache_file=True,
                                                   num_proc=80, remove_columns=['item', 'address', 'title', 'article', 'summary'])

valid_dataset = data['validation'].shuffle(seed=36).map(map_to, batched=True, batch_size=1000, drop_last_batch=True, num_proc=10, cache_file_name='validation.pkl', load_from_cache_file=True,
                                                        remove_columns=['item', 'address', 'title', 'article', 'summary'])

Loading cached processed dataset at train_*_of_00080.pkl
Loading cached processed dataset at validation_*_of_00010.pkl


In [14]:
data_collator = DataCollatorForSeq2Seq(tokenizer,
                                       model=model,
                                       padding=True,
                                       return_tensors='tf',
                                       label_pad_token_id=tokenizer.pad_token_id)

In [15]:
tf_train = model.prepare_tf_dataset(train_dataset,
#                                     collate_fn=data_collator,
                                    drop_remainder=True,
                                    batch_size=64,
                                    shuffle=True)

tf_dev = model.prepare_tf_dataset(valid_dataset,
                                    drop_remainder=True,
                                    batch_size=64,
                                    shuffle=True)

In [16]:
summary = model.fit(tf_train, validation_data=tf_dev, epochs=20, steps_per_epoch=100)

Epoch 1/20


2023-07-07 21:15:28.852962: E tensorflow/core/grappler/optimizers/meta_optimizer.cc:954] model_pruner failed: INVALID_ARGUMENT: Graph does not contain terminal node AssignAddVariableOp.
2023-07-07 21:15:30.409901: E tensorflow/core/grappler/optimizers/meta_optimizer.cc:954] model_pruner failed: INVALID_ARGUMENT: Graph does not contain terminal node AssignAddVariableOp.


100/100 [==============================] - ETA: 0s - loss: 1.7715 - perplexity: 5.8795

2023-07-07 21:17:12.894875: E tensorflow/core/grappler/optimizers/meta_optimizer.cc:954] model_pruner failed: INVALID_ARGUMENT: Graph does not contain terminal node AssignAddVariableOp.
2023-07-07 21:17:13.107872: E tensorflow/core/grappler/optimizers/meta_optimizer.cc:954] model_pruner failed: INVALID_ARGUMENT: Graph does not contain terminal node AssignAddVariableOp.


100/100 [==============================] - 185s 564ms/step - loss: 1.7715 - perplexity: 5.8795 - val_loss: 0.0626 - val_perplexity: 1.0646
Epoch 2/20
100/100 [==============================] - 42s 420ms/step - loss: 0.0441 - perplexity: 1.0451 - val_loss: 0.0153 - val_perplexity: 1.0154
Epoch 3/20
100/100 [==============================] - 42s 420ms/step - loss: 0.0184 - perplexity: 1.0185 - val_loss: 0.0047 - val_perplexity: 1.0047
Epoch 4/20
100/100 [==============================] - 42s 417ms/step - loss: 0.0104 - perplexity: 1.0105 - val_loss: 0.0026 - val_perplexity: 1.0026
Epoch 5/20
100/100 [==============================] - 42s 421ms/step - loss: 0.0067 - perplexity: 1.0067 - val_loss: 0.0017 - val_perplexity: 1.0017
Epoch 6/20
100/100 [==============================] - 42s 420ms/step - loss: 0.0049 - perplexity: 1.0050 - val_loss: 0.0012 - val_perplexity: 1.0012
Epoch 7/20
100/100 [==============================] - 42s 419ms/step - loss: 0.0037 - perplexity: 1.0037 - val_loss:

In [18]:
out_dir = 'sba3b3/tf_v1'
model.save_pretrained(out_dir)
tokenizer.save_pretrained(out_dir)

('sba3b3/tf_v1/tokenizer_config.json',
 'sba3b3/tf_v1/special_tokens_map.json',
 'sba3b3/tf_v1/sentencepiece.bpe.model',
 'sba3b3/tf_v1/added_tokens.json',
 'sba3b3/tf_v1/tokenizer.json')

In [20]:
out_dir = 'sba3b3/pt_v1'
model.save_pretrained(out_dir, safe_serialization=True)
tokenizer.save_pretrained(out_dir)

('sba3b3/pt_v1/tokenizer_config.json',
 'sba3b3/pt_v1/special_tokens_map.json',
 'sba3b3/pt_v1/sentencepiece.bpe.model',
 'sba3b3/pt_v1/added_tokens.json',
 'sba3b3/pt_v1/tokenizer.json')

In [21]:
!ls sba3b3/pt_v1

config.json		sentencepiece.bpe.model  tokenizer_config.json
generation_config.json	special_tokens_map.json
model.safetensors	tokenizer.json


In [22]:
# !rm -r sba3b3/pt_v1